# ReKIS dataset introduction
Inspired by: https://github.com/podondra/downscaling/blob/main/notebooks/01_reprojection.ipynb

In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib
import cartopy
import rasterio

# ReKIS
Load ReKIS

In [ ]:
CLIMATE_VARIABLE = 'TM'
YEAR = 2000
FILE_PATH = f'/home/tomas/ctu/current/rci_data/climate/ReKIS/KlimRefDS_v3.1_1961-2023/Raster/Tag/GK4/{CLIMATE_VARIABLE}/{CLIMATE_VARIABLE}_*.nc'
x = xr.open_mfdataset(FILE_PATH, decode_coords="all")
x = x[CLIMATE_VARIABLE]
x

Example

In [ ]:
date = f'2000-02-01'
date_tm = x.sel({'time': date})
date_tm.plot(cmap='coolwarm')
plt.show()

In [ ]:
images = [x.sel({'time' : f'2000-{month}-01'}) for month in range(1, 12, 3)]
vmax = max([i.max().values.item() for i in images])
vmin = min([i.min().values.item() for i in images])

In [ ]:
fig = plt.figure(figsize=(10, 8))

axs = []
p = None

for i, image in enumerate(images):
    axs.append(fig.add_subplot(2, 2, i + 1, projection=cartopy.crs.UTM(33)))
    p = image.plot(ax=axs[i], center = 0, cmap = 'coolwarm', vmin=vmin, vmax=vmax,
                   transform=cartopy.crs.epsg(31468), add_colorbar=False)
    axs[i].set_title(str(image.coords['time'].values)[:10])

for ax in axs:
    ax.add_feature(cartopy.feature.RIVERS)
    ax.add_feature(cartopy.feature.COASTLINE)
    ax.add_feature(cartopy.feature.BORDERS)
    ax.add_feature(cartopy.feature.STATES, linestyle=':')

cbar = fig.colorbar(
    p,
    label = 'Mean temperature [°C]',
    ax=axs,
    orientation='vertical',
    fraction=0.046,
    pad=0.04,
)

Examples in more days

In [ ]:
fig = plt.figure(figsize=(15, 8))
ax0 = fig.add_subplot(1, 2, 1, projection=cartopy.crs.epsg(31468))
date_tm.plot(ax = ax0, center = 0, cmap = 'coolwarm')

ax1 = fig.add_subplot(1, 2, 2, projection=cartopy.crs.UTM(33))
date_tm.plot(ax = ax1, center = 0, cmap = 'coolwarm', transform = cartopy.crs.epsg(31468))

for ax in [ax0, ax1]:
    ax.add_feature(cartopy.feature.RIVERS)
    ax.add_feature(cartopy.feature.COASTLINE)
    ax.add_feature(cartopy.feature.BORDERS)
    ax.add_feature(cartopy.feature.STATES, linestyle=':')

projections, we will use the right one

In [ ]:
fig = plt.figure(figsize=(10, 8))

ax = fig.add_subplot(1, 1, 1, projection=cartopy.crs.UTM(33))

img = x.sel({'time' : '2000-02-01'})
vmin = img.min().values.item()
vmax = img.max().values.item()

img.plot(ax = ax, center = 0, cmap = 'coolwarm', vmin=vmin, vmax=vmax, transform = cartopy.crs.epsg(31468), cbar_kwargs={'label': '°C'}, rasterized=True)

ax.add_feature(cartopy.feature.COASTLINE)
ax.add_feature(cartopy.feature.BORDERS)
ax.add_feature(cartopy.feature.STATES, linestyle=':')
ax.set_title('')
plt.show()

fig.savefig('images/rekis.pdf', bbox_inches='tight', dpi=300)

In [ ]:
mean_years = (x.sel({'time' : slice('1961', '2002')}).groupby('time.year')
              .mean(dim=['time', 'easting', 'northing']).compute())
mean_years

In [ ]:
plt.plot(range(1961, 2003), mean_years)
plt.title('Mean temperature in the ReKIS region through the years')
plt.show()

Mean temperature changing by each year, showing an increasing trend

In [ ]:
mean_day = x.sel({'time' : slice('1961', '1992')}).mean(dim='time').compute()
mean_val = x.sel({'time' : slice('1961', '1992')}).mean().values.item()

In [ ]:
fig = plt.figure(figsize=(10, 8))

ax = fig.add_subplot(1, 1, 1, projection=cartopy.crs.UTM(33))

img = mean_day

vmin = img.min().values.item()
vmax = img.max().values.item()

img.plot(ax = ax, center = 0, cmap = 'coolwarm', vmin=vmin, vmax=vmax, transform = cartopy.crs.epsg(31468), cbar_kwargs={'label': 'Mean temperature [°C]'}, rasterized=True)

ax.add_feature(cartopy.feature.COASTLINE)
ax.add_feature(cartopy.feature.BORDERS)
ax.add_feature(cartopy.feature.STATES, linestyle=':')
ax.set_title('Mean temperature through 1961-1992')
plt.show()

fig.savefig('images/rekis_mean.pdf', bbox_inches='tight', dpi=300)

Mean temperature through the years in each pixel, even in the mean, the high resolution features near the mountainous regions are still present.

In [ ]:
x.rio.set_spatial_dims("easting", "northing")
x.rio.crs, x.rio.width, x.rio.height, x.rio.resolution()

# CORDEX

In [ ]:
FILE_PATH_CORDEX = f'/home/tomas/ctu/current/rci_data/climate/CORDEX/tas/tas_EUR-11_ECMWF-ERAINT_evaluation_r1i1p1_GERICS-REMO2015_v1_day_*.nc'

In [ ]:
cordex = xr.open_mfdataset(
    FILE_PATH_CORDEX,
    decode_coords="all",
)
cordex = cordex["tas"]
cordex

In [ ]:
print(cordex.rio.crs)

In [ ]:
cordex_example = cordex.sel(time='2000-02-01' + "T12:00:00")

# it is in K
cordex_example -= 273.15
cordex_example

In [ ]:
cordex_example.plot()
plt.show()

Rotated pole projection

In [ ]:
cordex["rotated_latitude_longitude"].attrs

In [ ]:
rotated_pole_crs = cartopy.crs.RotatedPole(
    pole_latitude=cordex["rotated_latitude_longitude"].attrs[
        "grid_north_pole_latitude"
    ],
    pole_longitude=cordex["rotated_latitude_longitude"].attrs[
        "grid_north_pole_longitude"
    ],
)
fig, ax = plt.subplots(subplot_kw=dict(projection=rotated_pole_crs))
cordex_example.plot(ax=ax)
ax.add_feature(cartopy.feature.BORDERS)
ax.add_feature(cartopy.feature.STATES, linestyle=":")
ax.add_feature(cartopy.feature.RIVERS)
ax.add_feature(cartopy.feature.COASTLINE)

# v stupnoch Celzia

In [ ]:
cordex.rio.crs, cordex.rio.width, cordex.rio.height, cordex.rio.resolution()

reprojection with resampling

In [ ]:
# reproject cordex to target projection
# so that we can determine the resolution of input
# https://rasterio.readthedocs.io/en/latest/api/rasterio.enums.html#rasterio.enums.Resampling
RESAMPLING = rasterio.enums.Resampling.average
# https://corteva.github.io/rioxarray/stable/rioxarray.html#rioxarray.raster_array.RasterArray.reproject
cordex_resolution = cordex_example.rio.reproject(date_tm.rio.crs, resampling=RESAMPLING)
cordex.rio.resolution(), cordex_resolution.rio.resolution()

In [ ]:
fig, ax = plt.subplots(subplot_kw=dict(projection=cartopy.crs.UTM(33)))
cordex_resolution.plot(ax=ax, transform=cartopy.crs.epsg(31468))
ax.add_feature(cartopy.feature.BORDERS)
ax.add_feature(cartopy.feature.STATES, linestyle=":")
ax.add_feature(cartopy.feature.RIVERS)
ax.add_feature(cartopy.feature.COASTLINE)

In [ ]:
print(cordex_resolution)
print(cordex_resolution.rio.crs)

# Upscale ReKIS

Super-resolution methods usually need integer scale factors.
Therefore, we clip to box of 400 times 400 pixels and upscale to 10 km.
The result has 40 times 40 pixels.
Then, the scale factor is 10 (i.e. 2 times 5).

In [ ]:
y_example = date_tm.isel(northing=slice(0, 400), easting=slice(0, 400))
y_example.rio.set_spatial_dims("easting", "northing")
x_example = y_example.rio.reproject(
    y_example.rio.crs,
    resolution=(10_000, 10_000), # 10 x 10 km
    resampling=RESAMPLING,
)

fig = plt.figure(figsize=(16, 8))
ax0 = fig.add_subplot(1, 2, 1, projection = cartopy.crs.UTM(32))
y_example.plot(ax = ax0, transform = cartopy.crs.epsg(31468), cmap = "coolwarm")

ax1 = fig.add_subplot(1, 2, 2, projection = cartopy.crs.UTM(32))
x_example.plot(ax = ax1, transform = cartopy.crs.epsg(31468), cmap = 'coolwarm')

In [ ]:
(
    y_example.shape,
    x_example.shape,
    y_example.rio.resolution(),
    x_example.rio.resolution(),
    cordex_resolution.rio.resolution(),
    y_example.rio.bounds(),
    x_example.rio.bounds(),
)

# Reproject and match

In [ ]:
# https://corteva.github.io/rioxarray/stable/examples/reproject_match.html
x_cordex_example = cordex_example.rio.reproject_match(x_example, resampling=RESAMPLING)
(
    x_example.rio.shape,
    x_cordex_example.rio.shape,
    x_example.rio.resolution(),
    x_cordex_example.rio.resolution(),
    x_example.rio.bounds(),
    x_cordex_example.rio.bounds(),
)

In [ ]:
fig, axes = plt.subplots(ncols=2, subplot_kw=dict(projection=cartopy.crs.UTM(33)), figsize = (15, 8))
minimum = min(x_example.min(), x_cordex_example.min())
maximum = max(x_example.max(), x_cordex_example.max())
norm = matplotlib.colors.Normalize(vmin=minimum, vmax=maximum)
im = x_example.plot(
    ax=axes[0],
    cmap="coolwarm",
    center=0,
    transform=cartopy.crs.epsg(31468),
    add_colorbar=False,
    norm=norm,
)
x_cordex_example.plot(
    ax=axes[1],
    cmap="coolwarm",
    center=0,
    transform=cartopy.crs.epsg(31468),
    add_colorbar=False,
    norm=norm,
)
fig.colorbar(im, ax=axes)
axes[0].set_title("ReKIS")
axes[1].set_title("CORDEX")
for ax in axes:
    ax.add_feature(cartopy.feature.BORDERS)
    ax.add_feature(cartopy.feature.STATES, linestyle=":")
    ax.add_feature(cartopy.feature.RIVERS)
    ax.add_feature(cartopy.feature.COASTLINE)